# Gradient Boosting Machines (GBM)

Gradient Boosting builds a prediction function $F(x)$ **additively** by starting from a single constant and then fitting many small trees to the "leftover" errors. First you choose

$$
F_0(x) = \arg\min_{\gamma}\sum_{i=1}^n\ell(y_i,\gamma),
$$

i.e. the best constant predictor (mean of $y_i$ for squared‐error, log-odds for logistic loss).

<br>

<p align="center">
<img src="visualizations/GBM.png" width="600">
</p>

## Mathematical Foundation

At each iteration $m=1,\dots,M$:

**1. Compute pseudo-residuals**

$$
r_{i,m} = -\left.\frac{\partial}{\partial F(x_i)}\ell(y_i,F(x_i))\right|_{F=F_{m-1}},
$$

which generalizes the usual "residual" $y_i - \hat y_i$ to any differentiable loss. For squared error, $r_{i,m}=y_i-F_{m-1}(x_i)$; for logistic, $r_{i,m}=y_i - p_{i,m-1}$ with $p=\sigma(F)$.

**2. Fit a small regression tree** $h_m(x)$ to $\{(x_i,r_{i,m})\}$. This partitions the input space into regions $R_{j,m}$, $j=1,\dots,J$.

**3. Compute the per-leaf updates**

$$
\gamma_{j,m} = \arg\min_{\gamma}\sum_{x_i\in R_{j,m}} \ell(y_i, F_{m-1}(x_i)+\gamma),
$$

i.e. the optimal constant offset in each leaf. In squared loss $\gamma_{j,m}$ is just the mean of the residuals in leaf $j$; in logistic loss it's a Newton-step ratio $\sum(y_i-p)/\sum[p(1-p)]$.

**4. Update the ensemble**

$$
F_m(x) = F_{m-1}(x) + \eta\sum_{j=1}^J \gamma_{j,m}\,\mathbf{1}\{x\in R_{j,m}\},
$$

where $\eta\in(0,1]$ is the learning rate (shrinkage).

Repeat until $m=M$, yielding

$$
F_M(x) = F_0(x) + \sum_{m=1}^M \eta\,h_m(x).
$$

## Advanced Topics

### Key Intuitions

* This is **gradient descent in function-space**: each tree $h_m$ is a "search direction" (fit to negative gradients), and each $\gamma_{j,m}$ is the "step size" (optimal line search) in that region.
* You never build one huge tree; instead you take **many tiny steps**, which controls overfitting and makes tuning (tree depth, $\eta$, subsampling) more effective.
* **Pseudo-residuals** tell you locally which way to adjust your prediction to reduce loss most quickly.
* $\gamma$ appears first as the global starting constant $F_0$, and thereafter as the per-leaf offset that best corrects the current model in that region.

By iterating "direction = pseudo-residuals, step size = $\gamma$, add tree," GBM converges to a strong learner even when each individual tree is very weak.

## Key Characteristics

### Advantages
* High predictive accuracy on diverse datasets
* Handles mixed data types (numerical and categorical)
* Built-in feature selection through tree splits
* Robust to outliers and missing values
* Can capture complex non-linear patterns

### Limitations
* Prone to overfitting (requires careful tuning)
* Sensitive to hyperparameters (learning rate, depth, etc.)
* Sequential training (cannot parallelize easily)
* Can be computationally expensive
* Less interpretable than single decision trees

### When to Use
* When high predictive accuracy is crucial
* For structured/tabular data with mixed feature types
* When you have time for hyperparameter tuning
* For complex patterns that linear models can't capture
* In machine learning competitions

In [1]:
import numpy as np
from tqdm import tqdm
from cifar10.cifar10_utils import get_all_data, get_test_data, extract_images_pca, normalize_data

import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(project_root)
from images.image_preprocessing import extract_raw_pixels, extract_color_histogram, extract_hog, extract_lbp

from sklearn.tree import DecisionTreeRegressor

In [2]:
class GradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators  # number of boosting rounds
        self.learning_rate = learning_rate  # shrinkage factor for updates
        self.max_depth = max_depth  # depth limit for each regression tree
        self.trees = []  # list of tree ensembles per class
        self.K = None  # number of classes
        self.F0 = None  # initial score (log-odds)

    def _softmax(self, F):
        # convert raw scores to probabilities
        e = np.exp(F - np.max(F, axis=1, keepdims=True))
        return e / np.sum(e, axis=1, keepdims=True)

    def fit(self, X, y):
        n, _ = X.shape
        self.K = np.max(y) + 1  # one-hot encoding
        Y = np.eye(self.K)[y]

        # initial prediction: log of class priors
        pi = np.mean(Y, axis=0)
        self.F0 = np.log(pi + 1e-9)
        F = np.tile(self.F0, (n, 1))  # expand to all samples

        # boosting iterations
        for m in tqdm(range(self.n_estimators), desc="Training GBM"):
            trees_m = []
            P = self._softmax(F)
            # fit a tree for each class based on residuals
            for k in range(self.K):
                residual = Y[:, k] - P[:, k]
                tree = DecisionTreeRegressor(max_depth=self.max_depth)
                tree.fit(X, residual)
                # update raw score with scaled tree predictions
                F[:, k] += self.learning_rate * tree.predict(X)
                trees_m.append(tree)
            self.trees.append(trees_m)

    def predict_prob(self, X):
        n = X.shape[0]
        # start from initial score for each class
        F = np.tile(self.F0, (n, 1))
        # accumulate updates from all trees
        for trees_m in self.trees:
            for k, tree in enumerate(trees_m):
                F[:, k] += self.learning_rate * tree.predict(X)
        return self._softmax(F)

    def predict(self, X):
        # choose class with highest probability
        proba = self.predict_prob(X)
        return np.argmax(proba, axis=1)

In [7]:
# Load data
x_train, y_train = get_all_data()
x_test, y_test = get_test_data()

# Preprocess: extract HOG features, apply PCA and normalize
x_train_pca, pca = extract_images_pca(extract_hog(x_train))
x_train_norm, mean, std = normalize_data(x_train_pca)
x_test_norm = normalize_data(pca.transform(extract_hog(x_test)), mean, std)

# Train algorithm
gbm = GradientBoostingClassifier(learning_rate=0.1, n_estimators=10, max_depth=4)
gbm.fit(x_train_norm, y_train)

# Evaluate
from sklearn.metrics import accuracy_score

y_pred = gbm.predict(x_test_norm)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy: {acc:.4f}")

Training GBM: 100%|██████████| 10/10 [01:21<00:00,  8.14s/it]

Test accuracy: 0.3656
